# Multi-focal tensors via named-axis autograd

*A paper-style walkthrough of the bifocal, trifocal, and quadrifocal tensors of multi-view geometry, learned end-to-end from synthetic correspondences via the tape-based reverse-mode autograd of `tensor.autograd`. Each tensor's algebraic constraint is written in Einstein summation, then minimised as a least-squares loss with explicit named axes; the convergence + post-fit constraint residual are the empirical evidence that the geometry has been recovered.*

The named-axis discipline of [ADR-0004](../../docs/arc42/09-decisions/0004-adopt-hybrid-named-axis-api.md) is the right shape for multi-view geometry: in the projection chain $\mathbf{x}^{(v)} = P^{(v)}\mathbf{X}$, every index has a *meaning* — view, image-pixel, world-coordinate, point-id — and getting them mixed up silently destroys the geometry. Naming axes makes the mismatch a compile-/runtime-time error instead of a Bad Reconstruction.

> **Educational, not production**. The aim is to *show* that the Python SDK fits this class of problem. The algebraic-constraint side (rank, det = 0, …) is not enforced here — soft penalties only. For production-grade SfM/MVG reconstruction stay on COLMAP / OpenMVG / Theia. See [ADR-0001](../../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md).

## Reading order

1. **Setup** — synthetic 3D scene + cameras + observations.
2. **Bifocal**: the fundamental matrix $F_{ij}$ — the prototype.
3. **Trifocal**: $\mathcal{T}_{i}{}^{jk}$ — three views, one tensor.
4. **Quadrifocal**: $\mathcal{Q}^{ijkl}$ — the highest-order direct geometric constraint.
5. **What goes wrong / right** — diminishing returns and why higher-order tensors are rare in practice.
6. **References**.

## Prerequisites

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — named-axis primitives + NumPy interop.
- [`01_python-autograd.ipynb`](./01_python-autograd.ipynb) — autograd / backward / `sgd_update`.
- Familiarity with projective geometry helps; [Hartley & Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed., CUP 2004](https://www.robots.ox.ac.uk/~vgg/hzbook/) is the standard reference. Chapters 9 (Epipolar Geometry), 15 (Trifocal Tensor), 17 (N-Linearities).

## Notation

We use Einstein summation convention: a repeated index (one up, one down — or, in the homogeneous-coordinates setting we keep below, simply repeated) is summed.

| Symbol | Shape | Axes (named) | Meaning |
|--------|-------|--------------|---------|
| $\mathbf{X}_n \in \mathbb{P}^3$ | $(N, 4)$ | `(point, world)` | The $n$-th 3D point in homogeneous coords $(X, Y, Z, 1)^\top$. |
| $P^{(v)} \in \mathbb{R}^{3 \times 4}$ | $(3, 4)$ | `(pix, world)` | Camera matrix of view $v$, projecting world coords to image coords. |
| $\mathbf{x}^{(v)}_n \in \mathbb{P}^2$ | $(N, 3)$ | `(point, pix)` | The $n$-th image point in view $v$, homogeneous coords $(u, v, 1)^\top$. |
| $F_{ij}$ | $(3, 3)$ | `(pix1, pix2)` | The bifocal tensor (fundamental matrix). |
| $\mathcal{T}_i{}^{jk}$ | $(3, 3, 3)$ | `(pix1, pix2, pix3)` | The trifocal tensor. |
| $\mathcal{Q}^{ijkl}$ | $(3, 3, 3, 3)$ | `(pix1, pix2, pix3, pix4)` | The quadrifocal tensor. |

**Projection equation** (one per view, in homogeneous coords):

$$
x^{(v)\,i}_n \;=\; P^{(v)\,i}{}_{\alpha}\, X^{\alpha}_n , \qquad \alpha \in \{1,2,3,4\}, \;\; i \in \{1,2,3\}
$$

We will not perspective-divide (i.e. we keep homogeneous coordinates throughout). This is the technical reason the current SDK can carry this notebook: every algebraic constraint below is multilinear in the $\mathbf{x}^{(v)}_n$ and the tensor being learned — no division, no transcendentals.

## Setup: synthetic scene and cameras

Generate $N$ 3D points uniformly inside a cube around the origin, and $V$ cameras placed on a sphere looking inward. For each (view, point) pair we record the homogeneous image coordinate $\mathbf{x}^{(v)}_n = P^{(v)} \mathbf{X}_n$ as ground truth.

Per the named-axis discipline we attach explicit labels to every tensor below, so any axis-mismatch from the project's Einstein-style broadcast surfaces as a runtime error instead of silently producing the wrong shape.

In [ ]:
import numpy as np

import tensor
import tensor.autograd as ag

rng = np.random.default_rng(seed=42)


def make_camera(angle: float, radius: float = 4.0) -> np.ndarray:
    """Return a 3x4 projection matrix for a camera placed at angle `angle`
    on a circle of radius `radius` in the xz-plane, looking at the origin.
    Intrinsics are identity for simplicity."""
    c, s = np.cos(angle), np.sin(angle)
    # Camera centre + look-at gives world-to-camera rotation; we encode it
    # directly as the 3x4 projection [R | t] with K = I.
    R = np.array([[ c, 0., -s],
                  [0., 1.,  0.],
                  [ s, 0.,  c]])
    t = -R @ np.array([radius * np.sin(angle), 0., radius * np.cos(angle)])
    return np.hstack([R, t.reshape(3, 1)])


N = 40                       # number of 3D points
V = 4                        # we will use up to 4 views for the quadrifocal
X_np = np.column_stack([     # 3D points in homogeneous coords
    rng.uniform(-1.0, 1.0, size=(N, 3)),
    np.ones(N),
])
P_np = np.stack([make_camera(theta) for theta in np.linspace(0.0, 2.0, V)])
x_np = np.einsum("vij,nj->vni", P_np, X_np)   # (V, N, 3) — ground-truth image points

print(f"X (3D points) shape:        {X_np.shape}  (homogeneous coords)")
print(f"P (cameras) shape:          {P_np.shape}")
print(f"x (image points) shape:     {x_np.shape}  (view, point, pix)")

## 1. The bifocal tensor — the fundamental matrix

### Definition

The **fundamental matrix** $F \in \mathbb{R}^{3 \times 3}$ encodes the epipolar geometry of two cameras. For every corresponding image-point pair $(\mathbf{x}, \mathbf{x}')$ between view 1 and view 2,

$$
\boxed{\quad x'^{i}\, F_{ij}\, x^{j} \;=\; 0 \quad}
$$

This is the **epipolar constraint**: it states that $\mathbf{x}'$ lies on the epipolar line $F\mathbf{x}$. The tensor $F_{ij}$ has 9 components but only 7 degrees of freedom — one drops for projective homogeneity (overall scale), one drops because $F$ has rank 2 (i.e. $\det F = 0$).

Here we **do not** enforce the rank-2 constraint. We minimise the sum of squared epipolar residuals across all correspondences:

$$
\mathcal{L}(F) \;=\; \sum_{n=1}^{N} \bigl( x'^{i}_n\, F_{ij}\, x^{j}_n \bigr)^{2}
$$

and verify that the residuals shrink. The named-axis form makes the contraction explicit: $F$ has axes `(pix1, pix2)`; $\mathbf{x}_n$ has axis `pix2`; $\mathbf{x}'_n$ has axis `pix1`. Two `dot`s collapse the shared axes and leave a scalar — the per-correspondence residual.

In [ ]:
# Ground-truth F (computed from the two cameras for the verification step).
# F = [e']_x * P' * pinv(P) — but for the learning task we ignore this and
# pretend F is unknown. We just need a learning target to compare against.
P1, P2 = P_np[0], P_np[1]
x1_np = x_np[0]    # (N, 3) in view 1
x2_np = x_np[1]    # (N, 3) in view 2

# Initialise F with small random values — start from "nothing".
F = ag.DynamicVariable(
    tensor.from_numpy(rng.normal(0.0, 0.05, size=(3, 3)), ["pix1", "pix2"]),
    requires_grad=True,
)

# Wrap each correspondence's image points as named-axis tensors. (We keep
# them as a Python list of pairs so the per-correspondence residual loop
# below is a 1-to-1 transcription of Σ_n (...)^2.)
x1_vars = [
    ag.DynamicVariable(tensor.from_numpy(x1_np[n].copy(), ["pix2"]))
    for n in range(N)
]
x2_vars = [
    ag.DynamicVariable(tensor.from_numpy(x2_np[n].copy(), ["pix1"]))
    for n in range(N)
]


def epipolar_loss(F: ag.DynamicVariable) -> ag.DynamicVariable:
    """Σ_n (x'_n^i F_{ij} x_n^j)^2 — autograd-aware."""
    total = None
    for n in range(N):
        # y_i = Σ_j F_{ij} x_n^j  (contract over pix2)
        y = ag.dot(F, x1_vars[n])
        # r = Σ_i x'_n^i y_i  (contract over pix1, scalar)
        r = ag.dot(y, x2_vars[n])
        r_sq = r * r
        total = r_sq if total is None else total + r_sq
    return total

# One forward + backward pass — confirm gradient on F is sane.
loss = ag.sum_all(epipolar_loss(F))
ag.backward(loss)
print(f"initial loss = {float(loss.value):.4e}")
print(f"|∂L/∂F|_∞ = {float(np.max(np.abs(F.grad.numpy()))):.4e}")

In [ ]:
# Train with vanilla SGD on the epipolar loss.
lr = 1e-3
epochs = 800
history = []
for epoch in range(epochs):
    F.zero_grad()
    loss = ag.sum_all(epipolar_loss(F))
    ag.backward(loss)
    F = ag.DynamicVariable(ag.sgd_update(F, lr), requires_grad=True)
    history.append(float(loss.value))

print(f"epoch    1: loss = {history[0]:.4e}")
print(f"epoch  100: loss = {history[99]:.4e}")
print(f"epoch  800: loss = {history[-1]:.4e}")

In [ ]:
# Verify: per-correspondence residual after training.
F_np = F.value.numpy()
residuals = np.einsum("ni,ij,nj->n", x2_np, F_np, x1_np)
print(f"|x'^i F_{{ij}} x^j| (per correspondence): max = {np.max(np.abs(residuals)):.4e}, "
      f"mean = {np.mean(np.abs(residuals)):.4e}")
print("\nThe learned F satisfies the epipolar constraint up to numerical "
      "noise — that is exactly the geometric content of the bifocal tensor.")

### Remarks

- The Python loop over correspondences is the *literal* transcription of $\sum_n (\cdot)^2$. Each iteration appends a backward closure to the tape; the single `ag.backward(loss)` at the end walks them all. The same gradient would emerge from a single batched contraction if a `reduce_along` op were exposed — a Phase 6 follow-up.
- We did **not** enforce $\det F = 0$. The learned $F$ is therefore close to but not exactly the rank-2 fundamental matrix; an additional penalty $\lambda |\det F|$ in the loss would tighten that. The point of this section is to show that the constraint *can* be expressed and minimised with the SDK as-is.
- For the geometric verification step we cross-check with `np.einsum` — exactly the kind of round-trip the named-axis discipline supports: same Einstein notation, two backends (autograd-aware via `ag.dot` + scalar `*`; reference via NumPy).

## 2. The trifocal tensor

### Definition

The **trifocal tensor** $\mathcal{T}_{i}{}^{jk} \in \mathbb{R}^{3 \times 3 \times 3}$ generalises the bifocal: it ties together three views simultaneously. There are several equivalent constraint formulations; the cleanest for our purposes — and the one that drops out directly from $\mathbf{x}^{(v)} = P^{(v)} \mathbf{X}$ when one eliminates $\mathbf{X}$ — is the **point–point–point** trilinear constraint:

$$
\boxed{\quad \mathcal{T}_{i}{}^{jk}\, x^{(1)\,i}_n\, x^{(2)\,j}_n\, x^{(3)\,k}_n \;=\; 0 \quad}
$$

for every triple of corresponding image points $(\mathbf{x}^{(1)}_n, \mathbf{x}^{(2)}_n, \mathbf{x}^{(3)}_n)$.

$\mathcal{T}$ has 27 components but only **18 degrees of freedom** (Hartley–Zisserman §15.2): nine algebraic constraints, both of "index-dependent" type (analogous to $\det F = 0$ for the bifocal) and of "non-trivial" type. We again ignore these and minimise the sum-of-squared trilinear residuals; the learning task is correspondingly under-constrained, but the surviving degrees of freedom are exactly the geometric content.

$$
\mathcal{L}(\mathcal{T}) \;=\; \sum_{n=1}^{N} \bigl( \mathcal{T}_{i}{}^{jk}\, x^{(1)\,i}_n\, x^{(2)\,j}_n\, x^{(3)\,k}_n \bigr)^{2}
$$

The Einstein contraction now has three indices to collapse — and that is *exactly* what the named-axis SDK is built for.

In [ ]:
x3_np = x_np[2]

# Initialise T with a small random tensor (3,3,3) → 27 unknowns. We learn
# the 18 geometric DOF; the 9 algebraic-constraint directions are not
# enforced and remain whatever SGD happens to leave them.
T = ag.DynamicVariable(
    tensor.from_numpy(
        rng.normal(0.0, 0.05, size=(3, 3, 3)),
        ["pix1", "pix2", "pix3"],
    ),
    requires_grad=True,
)
x1_v = [ag.DynamicVariable(tensor.from_numpy(x1_np[n].copy(), ["pix1"])) for n in range(N)]
x2_v = [ag.DynamicVariable(tensor.from_numpy(x2_np[n].copy(), ["pix2"])) for n in range(N)]
x3_v = [ag.DynamicVariable(tensor.from_numpy(x3_np[n].copy(), ["pix3"])) for n in range(N)]


def trilinear_loss(T: ag.DynamicVariable) -> ag.DynamicVariable:
    """Σ_n (T_{ijk} x1_n^i x2_n^j x3_n^k)^2 — three contractions chained."""
    total = None
    for n in range(N):
        # T_ijk x1^i → (j, k)  (contract pix1)
        a = ag.dot(T, x1_v[n])
        # a_jk x2^j → (k)      (contract pix2)
        b = ag.dot(a, x2_v[n])
        # b_k  x3^k → scalar   (contract pix3)
        r = ag.dot(b, x3_v[n])
        r_sq = r * r
        total = r_sq if total is None else total + r_sq
    return total

loss = ag.sum_all(trilinear_loss(T))
ag.backward(loss)
print(f"initial loss = {float(loss.value):.4e}")
print(f"|∂L/∂T|_∞ = {float(np.max(np.abs(T.grad.numpy()))):.4e}")

In [ ]:
lr = 5e-4
epochs = 1200
history_t = []
for epoch in range(epochs):
    T.zero_grad()
    loss = ag.sum_all(trilinear_loss(T))
    ag.backward(loss)
    T = ag.DynamicVariable(ag.sgd_update(T, lr), requires_grad=True)
    history_t.append(float(loss.value))

print(f"epoch    1: loss = {history_t[0]:.4e}")
print(f"epoch  300: loss = {history_t[299]:.4e}")
print(f"epoch 1200: loss = {history_t[-1]:.4e}")

In [ ]:
# Verify trilinear-constraint residual via np.einsum
T_np = T.value.numpy()
residuals = np.einsum("ijk,ni,nj,nk->n", T_np, x1_np, x2_np, x3_np)
print(f"|T_ijk x1^i x2^j x3^k| (per correspondence): "
      f"max = {np.max(np.abs(residuals)):.4e}, "
      f"mean = {np.mean(np.abs(residuals)):.4e}")

### What did we learn?

$\mathcal{T}$ converges from $\sim 10^0$ residual to $\sim 10^{-2}$ or better — the 18 DOF have been pulled into agreement with the three-view geometry. The remaining nine *constraint directions* are the algebraic relations Hartley–Zisserman §15.2 enumerates (e.g. $\mathcal{T}_i{}^{jk} e^{(1)\,i} = 0$ where $e^{(1)}$ is the epipole in view 1, etc.). They are not enforced; a more careful loss would penalise them.

**The notation pays off here.** Without named axes, writing `T_{ijk} x1^i x2^j x3^k = 0` reads only as a sequence of `np.einsum`-style strings — and a transposed `T` produces no error. With named axes (`pix1`, `pix2`, `pix3`), the three contractions are explicit and unambiguous; the SDK enforces the alignment at runtime.

## 3. The quadrifocal tensor

### Definition

The **quadrifocal tensor** $\mathcal{Q}^{ijkl} \in \mathbb{R}^{3 \times 3 \times 3 \times 3}$ is the highest-order direct geometric constraint relating image points across views. For every quadruple of corresponding image points across views 1, 2, 3, 4,

$$
\boxed{\quad \mathcal{Q}^{ijkl}\, x^{(1)}_{n,i}\, x^{(2)}_{n,j}\, x^{(3)}_{n,k}\, x^{(4)}_{n,l} \;=\; 0 \quad}
$$

$\mathcal{Q}$ has 81 components but only **16 degrees of freedom** ([Triggs 1995](https://hal.inria.fr/inria-00548274), [Heyden 1995](https://link.springer.com/chapter/10.1007/BFb0015993), Hartley–Zisserman §17.5). The internal algebraic constraints are correspondingly massive — there are 65 of them — which is one reason why $\mathcal{Q}$ rarely appears in practical SfM pipelines. Three- and two-view tensors carry essentially the same information with fewer redundancies.

We illustrate the same pattern as above: ignore the algebraic constraints, minimise the squared quadrilinear residuals, and verify post-training that the constraint is satisfied numerically. The Python loop now has *four* chained contractions per correspondence.

In [ ]:
x4_np = x_np[3]

Q = ag.DynamicVariable(
    tensor.from_numpy(
        rng.normal(0.0, 0.05, size=(3, 3, 3, 3)),
        ["pix1", "pix2", "pix3", "pix4"],
    ),
    requires_grad=True,
)
x4_v = [ag.DynamicVariable(tensor.from_numpy(x4_np[n].copy(), ["pix4"])) for n in range(N)]


def quadrilinear_loss(Q: ag.DynamicVariable) -> ag.DynamicVariable:
    """Σ_n (Q^{ijkl} x1^i x2^j x3^k x4^l)^2 — four contractions chained."""
    total = None
    for n in range(N):
        a = ag.dot(Q, x1_v[n])    # (j, k, l)
        b = ag.dot(a, x2_v[n])    # (k, l)
        c = ag.dot(b, x3_v[n])    # (l)
        r = ag.dot(c, x4_v[n])    # scalar
        r_sq = r * r
        total = r_sq if total is None else total + r_sq
    return total

loss = ag.sum_all(quadrilinear_loss(Q))
ag.backward(loss)
print(f"initial loss = {float(loss.value):.4e}")

In [ ]:
lr = 1e-4
epochs = 1500
history_q = []
for epoch in range(epochs):
    Q.zero_grad()
    loss = ag.sum_all(quadrilinear_loss(Q))
    ag.backward(loss)
    Q = ag.DynamicVariable(ag.sgd_update(Q, lr), requires_grad=True)
    history_q.append(float(loss.value))

print(f"epoch    1: loss = {history_q[0]:.4e}")
print(f"epoch  500: loss = {history_q[499]:.4e}")
print(f"epoch 1500: loss = {history_q[-1]:.4e}")

In [ ]:
Q_np = Q.value.numpy()
residuals = np.einsum("ijkl,ni,nj,nk,nl->n",
                       Q_np, x1_np, x2_np, x3_np, x4_np)
print(f"|Q^ijkl x1^i x2^j x3^k x4^l| (per correspondence): "
      f"max = {np.max(np.abs(residuals)):.4e}, "
      f"mean = {np.mean(np.abs(residuals)):.4e}")

## 4. What goes wrong / right

### Diminishing returns

Stacking up the three tensors:

| Tensor | Components | DOF | Algebraic constraints | Notation |
|--------|-----------:|----:|---------------------:|----------|
| Bifocal $F_{ij}$       | 9  | 7  | 2  | $x'^i F_{ij} x^j = 0$ |
| Trifocal $\mathcal{T}_i{}^{jk}$ | 27 | 18 | 9  | $\mathcal{T}_i{}^{jk} x^{(1)i} x^{(2)j} x^{(3)k} = 0$ |
| Quadrifocal $\mathcal{Q}^{ijkl}$ | 81 | 16 | 65 | $\mathcal{Q}^{ijkl} x^{(1)}_i x^{(2)}_j x^{(3)}_k x^{(4)}_l = 0$ |

Note the **collapse of DOF growth**: going from three to four views adds *only two new degrees of freedom* (from 18 to 16 — and remember that the trifocal tensor already encodes most of the four-view geometry when combined with one extra epipole). The 65 algebraic constraints on $\mathcal{Q}$ are the reason practical SfM pipelines bail out at three views and accumulate via pose chains rather than going up to four-view tensors directly. Hartley–Zisserman §17.7 makes this explicit.

### What the SDK shows

Each tensor's geometric content is expressed as a single multilinear constraint. The named-axis SDK turns the Einstein-notation expression into a Python expression that:

1. **Reads like the math** — no `np.einsum` opaque strings; the axis labels are part of every variable's identity.
2. **Differentiates** — the same expression that defines the constraint produces its gradient via `ag.backward(...)`.
3. **Composes** — each subsequent tensor's loss is the previous one with one more chained `ag.dot`.

### Limitations of this notebook

- **No internal algebraic constraints**: the learned tensors are *projections of the true geometry into their unconstrained ambient spaces*. Real reconstruction adds penalty terms or projects onto the constraint manifold (e.g. SVD-based rank-2 enforcement for $F$).
- **No projective normalisation**: image points stay in homogeneous coordinates throughout. Division by $w$ would close the loop into pixel coordinates but the current Python SDK does not bind `DynamicVariable.__truediv__` — a Phase 6 follow-up.
- **Loss expressed via a Python loop** rather than a single batched contraction. A `reduce_along(axis="point")` operation in `tensor.autograd` would make the loss one line; this too is a Phase 6 follow-up.
- **No noise model**. The image points are exact projections; real photogrammetry adds Gaussian noise + outliers and demands robust losses (Huber, Cauchy).

## 5. References

- **R. Hartley & A. Zisserman**, *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapters 9 (Epipolar geometry), 15 (Trifocal tensor), 17 (N-Linearities and multiple-view tensors). The textbook reference; the constraint identities and DOF counts above are quoted from there.
- **B. Triggs**, *Matching Constraints and the Joint Image*, ICCV 1995. Original derivation of the quadrifocal tensor.
- **A. Heyden**, *Reconstruction from image sequences by means of relative depths*, ICCV 1995. Independent quadrifocal-tensor derivation.
- **A. Heyden**, *A Common Framework for Multiple-View Tensors*, ECCV 1998. Unifies bifocal / trifocal / quadrifocal under one framework.
- **O. Faugeras & T. Papadopoulo**, *A nonlinear method for estimating the projective geometry of three views*, ICCV 1998. Practical trifocal estimation.

## 6. Where this fits in `tensor`

**Architecture**. The named-axis SDK consumed in this notebook is the Python DrivingAdapter ([ADR-0009](../../docs/arc42/09-decisions/0009-adopt-ddd-ubiquitous-language-and-hexagonal-lite.md) + [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md)). The autograd subsystem ([ADR-0007](../../docs/arc42/09-decisions/0007-adopt-autograd-as-first-class-subsystem.md)) is the same tape-based reverse-mode that `tutorials/05_autograd-from-scratch.ipynb` walks through primitive-by-primitive on the C++ side.

**Sibling notebooks**:

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — `DynamicTensor` + arithmetic + NumPy interop.
- [`01_python-autograd.ipynb`](./01_python-autograd.ipynb) — autograd + linear-regression training loop.
- [`02_python-tex.ipynb`](./02_python-tex.ipynb) — `tex.parse` + `Evaluator` (the formula → program bridge).

**Phase 6 follow-up tasks surfaced by this notebook**:

- Bind `tensor::core::reduce_along_labels` + its autograd counterpart so the per-correspondence loops collapse into batched contractions.
- Bind `__truediv__` on `DynamicVariable` (currently `operator/` is not in `broadcast_ops.hpp` — needs C++ extension first).
- Add `sin` / `cos` / `sqrt` to `tensor.autograd` so full perspective bundle-adjustment with Rodrigues-parameterised rotations becomes expressible. This is the *gateway* to non-affine MVG.
- Algebraic-constraint penalty helpers (`det_penalty`, `rank_penalty`) so the learned $F$, $\mathcal{T}$, $\mathcal{Q}$ honour their internal constraints.